# F5-TTS Pipeline Worker — Colab GPU

Runs the **F5-TTS worker** on Colab's free GPU (T4). Unlike the Abogen worker,
this notebook does **not** need a separate Flask server — the worker synthesises
audio directly using F5-TTS zero-shot voice cloning.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

**Set Colab Secrets** (🔑 icon, left sidebar):

| Secret | Value |
|--------|-------|
| `NGROK_TOKEN` | ngrok authtoken (dashboard.ngrok.com → Your Authtoken) |
| `SSH_HOST` | server IP / hostname |
| `SSH_USER` | SSH username |
| `SSH_KEY`  | full private key (paste `-----BEGIN ... KEY-----`) |
| `SSH_REMOTE_PATH` | remote output dir, e.g. `/opt/tts-node/outputs` |
| `SSH_PORT` | SSH port (default 22) |
| `GITHUB_TOKEN` | PAT for private repos (optional) |
| `GITHUB_USER`  | GitHub username (optional) |

**Reference audio** is auto-detected from the F5-TTS package after install.
You can override it with your own WAV in cell 5.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Mount Google Drive (fallback output + reference audio storage) ─────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
os.makedirs('/content/ref', exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── 2. Secrets ────────────────────────────────────────────────────────────────
# Add these in the Secrets panel (🔑 icon, left sidebar):
#
#   GITHUB_TOKEN      → ghp_...  (Personal Access Token — private repos only)
#   GITHUB_USER       → your GitHub username
#   NGROK_TOKEN       → ngrok authtoken (dashboard.ngrok.com → Your Authtoken)
#
#   SSH_HOST          → server IP or hostname  e.g. 10.0.0.1
#   SSH_USER          → SSH username           e.g. root
#   SSH_KEY           → private key contents   (paste the full -----BEGIN ... KEY-----)
#   SSH_REMOTE_PATH   → remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          → SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'ngrok token:  {"ok" if NGROK_TOKEN else "MISSING"}')
print(f'SSH host:     {"ok" if SSH_HOST    else "MISSING"}')
print(f'SSH key:      {"ok" if SSH_KEY     else "MISSING"}')

In [ ]:
# ── 3. SSH setup + mount remote output dir via sshfs ─────────────────────────
import os, subprocess, textwrap, re

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)

def _reconstruct_pem(raw: str) -> str:
    key = raw.replace('\\n', '\n').strip()
    pattern = r'(-----BEGIN [^-]+-----)(.*?)(-----END [^-]+-----)'
    match = re.search(pattern, key, re.DOTALL)
    if match:
        header, body, footer = match.groups()
        body = ''.join(body.split())
        wrapped = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
        return f'{header}\n{wrapped}\n{footer}\n'
    return key + '\n'

with open(SSH_KEY_PATH, 'w') as f:
    f.write(_reconstruct_pem(SSH_KEY))
os.chmod(SSH_KEY_PATH, 0o600)

# Validate key format
check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
if check.returncode != 0:
    print("Debug - First 100 chars of processed key:\n", open(SSH_KEY_PATH).read()[:100])
    raise RuntimeError(f'SSH Key Error: {check.stderr}')

# Configure SSH client
with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
            ConnectTimeout 10
    """))

print(f'SSH key verified. Testing connection to {SSH_USER}@{SSH_HOST}...')

# Test connectivity and ensure remote path exists
test_conn = subprocess.run(['ssh', f'{SSH_USER}@{SSH_HOST}', f'mkdir -p {SSH_REMOTE_PATH}'], capture_output=True, text=True)
if test_conn.returncode != 0:
    raise RuntimeError(f'Connection failed: {test_conn.stderr}')

# Mount via sshfs
!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_cmd = [
    'sshfs', f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}', SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3'
]

result = subprocess.run(mount_cmd, capture_output=True, text=True)
if result.returncode == 0:
    OUTPUT_DIR = SSHFS_MOUNT
    os.environ['OUTPUT_DIR'] = OUTPUT_DIR
    print(f'Successfully mounted {SSH_REMOTE_PATH} to {SSHFS_MOUNT}')
else:
    print(f'Mount failed: {result.stderr}')

In [ ]:
# ── 4. Config — fill these in ─────────────────────────────────────────────────
REDIS_URL = 'rediss://redis.0xvictor.dev:6380'   # rediss:// = SSL/TLS (port 6380)
WORKER_ID = 'colab-f5-1'

# Reference audio defaults — overridden in cell 5 after F5-TTS is installed
F5_REF_AUDIO = '/content/ref/reference.wav'
F5_REF_TEXT  = 'Some call me nature, others call me mother nature.'

os.environ['REDIS_URL']       = REDIS_URL
os.environ['WORKER_ID']       = WORKER_ID
os.environ['OUTPUT_DIR']      = OUTPUT_DIR
os.environ['SPOOL_DIR']       = SPOOL_DIR
os.environ['USE_GPU']         = 'true'
os.environ['F5_DEVICE']       = 'cuda'
os.environ['F5_MODEL']        = 'F5TTS_v1_Base'
os.environ['F5_REF_AUDIO']    = F5_REF_AUDIO
os.environ['F5_REF_TEXT']     = F5_REF_TEXT
os.environ['F5_SPEED']        = '1.0'
os.environ['F5_MAX_CHARS']    = '1000'
os.environ['PROMETHEUS_PORT'] = '8000'
print('Config set.')

In [ ]:
# ── 5. Reference audio ────────────────────────────────────────────────────────
# F5-TTS ships a sample English reference WAV inside the package.
# We use that automatically so nothing needs uploading.
# To use your own voice instead, uncomment the upload block at the bottom.

import os, shutil, glob

os.makedirs('/content/ref', exist_ok=True)

def _find_bundled_sample():
    """Locate basic_ref_en.wav that ships with the f5_tts package."""
    try:
        import f5_tts
        pkg_root = os.path.dirname(f5_tts.__file__)
        candidates = glob.glob(os.path.join(pkg_root, '**', 'basic_ref_en.wav'), recursive=True)
        if candidates:
            return candidates[0]
    except Exception:
        pass
    return None

sample = _find_bundled_sample()
if sample:
    shutil.copy(sample, F5_REF_AUDIO)
    # The bundled sample's transcript (from F5-TTS repo)
    F5_REF_TEXT = "Some call me nature, others call me mother nature."
    os.environ['F5_REF_AUDIO'] = F5_REF_AUDIO
    os.environ['F5_REF_TEXT']  = F5_REF_TEXT
    print(f'Using bundled sample voice: {sample}')
    print(f'Ref text: "{F5_REF_TEXT}"')
else:
    print('Bundled sample not found — run install cell first, then re-run this cell.')

# ── Override: upload your own voice (optional) ────────────────────────────────
# Uncomment to replace the bundled sample with your own WAV (3–30s clean speech).
#
# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded:
#     with open(F5_REF_AUDIO, 'wb') as f:
#         f.write(uploaded[fname])
# F5_REF_TEXT = 'Paste the exact words spoken in your uploaded WAV here.'
# os.environ['F5_REF_AUDIO'] = F5_REF_AUDIO
# os.environ['F5_REF_TEXT']  = F5_REF_TEXT
# print(f'Custom voice loaded: {F5_REF_AUDIO}')

if os.path.exists(F5_REF_AUDIO):
    print(f'Reference audio ready ({os.path.getsize(F5_REF_AUDIO):,} bytes)')

In [ ]:
# ── 6. Install system dependencies ───────────────────────────────────────────
!apt-get install -y ffmpeg > /dev/null 2>&1
print('ffmpeg installed.')

In [ ]:
# ── 7. Install F5-TTS + GPU PyTorch ──────────────────────────────────────────
# Colab T4 runs CUDA 12.x — use the cu121 wheel
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q f5-tts soundfile pydub redis prometheus_client
print('All packages installed.')

In [ ]:
# ── 8. Fetch the F5-TTS worker script ────────────────────────────────────────
# Pulls worker.py directly from your repo (adjust URL for private repos)
import subprocess
result = subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/YOUR_ORG/YOUR_REPO',   # ← replace with your repo
    '/content/abogen'
], capture_output=True, text=True)
print(result.stdout or result.stderr)

import shutil
shutil.copy('/content/abogen/tts-node/f5-worker/worker.py', '/content/worker.py')
print('worker.py ready.')

In [ ]:
# ── 9. Verify Redis connection ────────────────────────────────────────────────
import redis
r = redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

In [ ]:
# ── 10. Warm up F5-TTS model (downloads weights ~1.2 GB on first run) ─────────
print('Loading F5-TTS model... (first run downloads ~1.2 GB, subsequent runs are instant)')
import sys
sys.path.insert(0, '/content')

_noop = lambda *args, **kwargs: None

# Trigger model download + load
from f5_tts.api import F5TTS
import os
_engine = F5TTS(
    model=os.environ['F5_MODEL'],
    device=os.environ['F5_DEVICE'],
)
print('Model loaded. Running test synthesis...')

# Quick smoke test
wav, sr, _ = _engine.infer(
    ref_file=F5_REF_AUDIO,
    ref_text=F5_REF_TEXT,
    gen_text='Pipeline worker ready.',
    show_info=_noop,
    progress=_noop,
)
print(f'Smoke test OK — generated {len(wav)/sr:.2f}s of audio at {sr} Hz')

In [ ]:
# ── 11. Start the F5-TTS worker ───────────────────────────────────────────────
# This cell runs until the Colab session ends or you interrupt it.
# The worker pulls chunk jobs from pipeline:tts and writes MP3s to OUTPUT_DIR.
%run /content/worker.py